# Downscale-Bench meta-postprocessing
---

### Introduction
The following Jupyter Notebook is designed to compare different downscaling solution against each other. It provides several plotting routines that allow for a systematic analysis, called meta-postprocessing, and enables the user to benchmark a new custom downscaling method against the baselines provided with *DownscaleBench*. <br>
All routines expect that the first postprocessing step, the single model evaluation, has been carried out beforehand. Thus, running `main_postprocess.py` as described in the repository's README is a mandatory prerequisite.

### Software requirements
* matplotlib/3.4.3 (version >= 3.4.3)
* cartopy/0.20.0 (version >= 0.20.0)
* skimage/0.18.3 (version >= 0.18.3)
* numpy (version >= 1.21.3)
* xarray (version >= 0.20.1)

## **Main**

We start by importing the required Python packages and routines:

In [ ]:
# Import modules
%matplotlib inline
import os, sys
base_dir = "../"
sys.path.extend([f"{base_dir}/postprocess"])

# for plotting
from metapostprocess import Config, score_line_plots, skill_box_plot, model_comparison_plot, spectra_plot

### Config parameters
 ---
The following cells rely on an instance of a `Config` class that is used to define parameters for our plotting routines. <br>
The following standard parameters are expected:
 
 - base_folder : define the absolute path location where all the results for all the models are stored e.g. ```"/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/results"```
 - variable : the target downscaling variable e.g. ```"t2m"``` (other options: "wind_speed","solar_irradiance")
 - models : list of competing models e.g. ```["deepru","sha_unet","sha_wgan","swinir"]```. The name of these models should exactly match with their corresponding folder in the base_folder.
 - seasons : If set to ```"year"```, the intercomparison is performed on the whole test dataset from 2018. Other options are "DJF","JJA","MAM","SON", corresponding to specific seasons.
 - metric : deinfe the evaluation metric e.g. ```rmse``` (evaluation metric must be available from the preceiding postprocessing-step)


The parameters can be either directly passed as arguments to the initialization of the `Config`-class or alternatively, as done in this notebook, be parsed from a dictionary. 

In [ ]:
#config parameters
config = {}
config['base_folder'] = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/results"
config['variable'] = "t2m"
config['models'] = ["sha_unet", "sha_wgan", "deepru","swinir"]
config['seasons'] = "year"   #["year","DJF","JJA","MAM","SON"]
config['metric'] = "rmse" #["bias","grad_amplitude","me_std","rmse"]                     

config = Config(**config)

### Plot metrics against daytime

The following cell plots the user-defined metric values from the competing models against the daytime. The plot therefore helps us to reveal daytime-depending deficiencies and strengths of the respective downscaling solutions.

In addition, a dictionary to control the labels of the y-axis should be parsed, e.g. to get the correct unit.

In [ ]:
#define metric
metric_name = f'{config.metric}' 
metric_unit = "K"
metric_dict = {metric_name.upper(): metric_unit}

#### Parameters of the routine

The `score_line_plots`-routine requires the following arguments:

- config : an instance of the Config place defined in cell before 
- uncertainty : Plot uncertainty range for models (set between the $1$ percentile to $99$ percentile of the block-bootstrapped scores) 
- metric_unit : metric dict as defined in the cell above.
- kwargs: Other keyword-arguments for plot_metric_line-routine

In [ ]:
score_line_plots(config, uncertainty=False, metric_dict=metric_dict, value_range=[0., 2.5], linestyle=["y-", "b-", "m-", "k-"])

### Skill score plots
--- 
Different model solutions can be directly compared in terms of skill scores. Here, we choose a box-plot for a compact visulaization. <br>
In the following, we overwrite specific keyword-value pairs of the `Config`-instance to change our settings.

#### Parameters of the routine

The `skill_box_plot`-routine requires the energy_plot(config,var_info)following arguments:

- config : an instance of the Config place defined in cell before 
- ref_model : reference model for skill score calculations (must be one of the models defined by the `models`-property of the `config`-instance)

In [ ]:
config.metric = "rmse"
config.models = ["sha_unet", "sha_wgan", "deepru", "swinir", "bilinear"]

skill_box_plot(config, ref_model="bilinear")

### Power Spectra Analysis
---
The following cell plots the radially averaged spectral power for all competing models as well as the ground truth data as a measure for spatial variability in the data. 

#### Parameters of the routine
- config : an instance of the Config place defined in cell before
- var_info: dictionary in the form of *{variable : unit}* to label the y-axis

In [ ]:
var_info = {config.variable : "K**2 m"}

spectra_plot(config, var_info, colors=["red", "blue", "dimgrey", "gold", "green", "black"])            

## Sample intercomparsion 

The following cell plots the downscaling results of all models for a specific sample from the test dataset (data from year 2018). 

#### Parameters of the routine
- config : an instance of the Config place defined in cell before
- datetime: Pass a datetime-string (readable by pandas) for a specific sample from 2018. For None, the averaging over the season will be performed.


In [ ]:
config.datetime = {"month" : "Mar" , 
                   "date" : 31,
                   "hour" : 14}

model_comparison_plot(config)